In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

`catalog` and `data_source` are declared as widgets rather than plain variables. Both appear as input fields at the top of the notebook, and both can be overridden by a Databricks Job at runtime.

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://energybars-dev-eu-west-3-raw/{data_source}/*.csv'
print(base_path)

s3://energybars-dev-eu-west-3-raw/customers/*.csv



## Bronze: raw ingest

Reads the CSV. No cleaning, no type changes beyond schema inference.


In [0]:
df = spark.read.format("csv").load(base_path)
display(df.limit(10))

_c0,_c1,_c2
customer_id,customer_name,city
789201,FitFuel Market,Bengaluru
789202,FitFuel Market,Hyderabad
789203,FitFuel Market,New Delhi
789301,Athlete's Choice Store,Bengaluru
789303,Athlete's Choice Store,New Delhi
789101,Endurance Foods,Bengalore
789102,Endurance Foods,Hyderabad
789103,Endurance Foods,New Delhi
789121,HydroBoost Nutrition,Hyderabad


Let's add a few options: header, interSchema.

Three metadata columns are added from `_metadata`:

| Column | Purpose |
|---|---|
| `read_timestamp` | when the row was ingested |
| `file_name` | which source file the row came from |
| `file_size` | sanity check on the source file |

These support lineage and debugging: when a bad value surfaces in a dashboard, the originating file can be identified immediately.

In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
# print check data type
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
display(df.limit(10))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-09-08T13:33:43.009Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-09-08T13:33:43.009Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-09-08T13:33:43.009Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-09-08T13:33:43.009Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-09-08T13:33:43.009Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-09-08T13:33:43.009Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-09-08T13:33:43.009Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-09-08T13:33:43.009Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-09-08T13:33:43.009Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-09-08T13:33:43.009Z,customers.csv,1404


Written in **overwrite** mode with **Change Data Feed** enabled. 
Each run receives a full snapshot, CDF so row-level changes are tracked and the table can be audited or time-travelled.

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")